In [1]:
import os
print(os.getcwd())

C:\Users\danye\Documents\Proyecto_GDIABD_Cybersecurity_Data_Analytics\Notebook


In [3]:
import os
os.environ["HADOOP_HOME"] = "C:\\Users\\danye\\Documents\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ProyectoFinal_Guia03")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

print("Spark:", spark.version)
print("Paralelismo por defecto:", spark.sparkContext.defaultParallelism)

C:\Users\danye\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark: 4.2.0
Paralelismo por defecto: 16


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ProyectoFinal_Guia03")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

print("Spark:", spark.version)
print("Paralelismo por defecto:", spark.sparkContext.defaultParallelism)

Spark: 4.2.0
Paralelismo por defecto: 16


In [5]:
ruta_parquet = Path("../Datos/Analiticos/Parquet/dataset_crudo_limpio_final.parquet")

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

df = spark.read.parquet(str(ruta_parquet.resolve()))

print("Filas:", df.count())
print("Columnas:", len(df.columns))
df.printSchema()

Filas: 2830540
Columnas: 84
root
 |-- flow_id: string (nullable = true)
 |-- source_ip: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- destination_ip: string (nullable = true)
 |-- destination_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- total_backward_packets: integer (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std:

In [6]:
from pyspark.sql import functions as F

#1 Duración de flujo no puede ser negativa
df_filtrado = df.filter(F.col("flow_duration") >= 0)

#2 Puertos dentro de rango válido (0-65535)
df_filtrado = df_filtrado.filter(
    (F.col("source_port").between(0, 65535)) &
    (F.col("destination_port").between(0, 65535))
)

#3 Conteos de paquetes no negativos
df_filtrado = df_filtrado.filter(
    (F.col("total_fwd_packets") >= 0) &
    (F.col("total_backward_packets") >= 0)
)

#4 Consistencia min <= mean <= max en longitud de paquete
df_filtrado = df_filtrado.filter(
    (F.col("min_packet_length") <= F.col("packet_length_mean")) &
    (F.col("packet_length_mean") <= F.col("max_packet_length"))
)

# 5. flow_bytes_s y flow_packets_s no pueden ser Infinity/NaN
df_filtrado = df_filtrado.filter(
    ~F.col("flow_bytes_s").isNull() &
    ~F.isnan(F.col("flow_bytes_s")) &
    (F.col("flow_bytes_s") != float("inf")) &
    ~F.col("flow_packets_s").isNull() &
    ~F.isnan(F.col("flow_packets_s")) &
    (F.col("flow_packets_s") != float("inf"))
)

print("Filas antes del filtrado:", df.count())
print("Filas después del filtrado:", df_filtrado.count())

Filas antes del filtrado: 2830540
Filas después del filtrado: 2827562


In [7]:
#1 Ratio de bytes enviados vs recibidos 
df_features = df_filtrado.withColumn(
    "ratio_fwd_bwd_bytes",
    F.when(F.col("total_length_of_bwd_packets") > 0,
           F.col("total_length_of_fwd_packets") / F.col("total_length_of_bwd_packets")
    ).otherwise(F.lit(0.0))
)

#2 Total de banderas TCP activadas por flujo
df_features = df_features.withColumn(
    "total_flags_activadas",
    F.col("fin_flag_count") + F.col("syn_flag_count") + F.col("rst_flag_count") +
    F.col("psh_flag_count") + F.col("ack_flag_count") + F.col("urg_flag_count")
)

#3 Categoría de duración del flujo 
df_features = df_features.withColumn(
    "categoria_duracion",
    F.when(F.col("flow_duration") < 1000, "Corto")
     .when(F.col("flow_duration") < 100000, "Medio")
     .otherwise("Largo")
)

df_features.select(
    "flow_id", "ratio_fwd_bwd_bytes", "total_flags_activadas", "categoria_duracion"
).show(10, truncate=False)

+--------------------------------------+-------------------+---------------------+------------------+
|flow_id                               |ratio_fwd_bwd_bytes|total_flags_activadas|categoria_duracion|
+--------------------------------------+-------------------+---------------------+------------------+
|192.168.10.3-192.168.10.9-88-1031-6   |1.1690821256038648 |1                    |Corto             |
|192.168.10.9-69.31.33.224-1057-80-6   |0.5392670157068062 |1                    |Medio             |
|192.168.10.3-192.168.10.9-88-1058-6   |1.0335120643431635 |1                    |Medio             |
|192.168.10.3-192.168.10.9-389-1060-6  |3.9441233140655108 |1                    |Medio             |
|192.168.10.3-192.168.10.17-88-35499-6 |1.0                |1                    |Medio             |
|192.168.10.3-192.168.10.17-88-35504-6 |0.9993654822335025 |1                    |Medio             |
|192.168.10.3-192.168.10.50-389-55724-6|0.5276073619631901 |1                    |

In [8]:
# Registrar la vista temporal
df_features.createOrReplaceTempView("trafico_red")

consulta = """
SELECT
    label,
    protocol,
    categoria_duracion,
    COUNT(*) AS total_flujos,
    ROUND(AVG(total_flags_activadas), 2) AS promedio_flags,
    ROUND(AVG(ratio_fwd_bwd_bytes), 4) AS promedio_ratio_fwd_bwd,
    ROUND(AVG(flow_bytes_s), 2) AS promedio_bytes_por_segundo
FROM trafico_red
GROUP BY label, protocol, categoria_duracion
ORDER BY total_flujos DESC
"""

resultado_sql = spark.sql(consulta)
resultado_sql.show(20, truncate=False)

+----------------+--------+------------------+------------+--------------+----------------------+--------------------------+
|label           |protocol|categoria_duracion|total_flujos|promedio_flags|promedio_ratio_fwd_bwd|promedio_bytes_por_segundo|
+----------------+--------+------------------+------------+--------------+----------------------+--------------------------+
|BENIGN          |6       |Largo             |608382      |1.15          |7.3871                |11571.25                  |
|BENIGN          |6       |Corto             |523099      |1.49          |0.5962                |5881272.35                |
|BENIGN          |17      |Medio             |476178      |0.0           |0.4755                |10835.16                  |
|BENIGN          |17      |Corto             |375025      |0.0           |0.4793                |2489296.91                |
|PortScan        |6       |Corto             |158193      |1.0           |0.1764                |220102.69                 |


In [9]:
resultado_sql.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [20]: [source_port#2, destination_port#4, protocol#5, flow_duration#7, total_fwd_packets#8, total_backward_packets#9, total_length_of_fwd_packets#10, total_length_of_bwd_packets#11, flow_bytes_s#20, flow_packets_s#21, min_packet_length#44, max_packet_length#45, packet_length_mean#46, fin_flag_count#49, syn_flag_count#50, rst_flag_count#51, psh_flag_count#52, ack_flag_count#53, urg_flag_count#54, label#83]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/danye/Documents/Proyecto_GDIABD_Cybersecurity_Data_Analytics/Datos/Analiticos/Parquet/dataset_crudo_limpio_final.parquet]
PushedFilters: [IsNotNull(flow_duration), IsNotNull(source_port), IsNotNull(destination_port), IsNotNull(total_fwd_packets

In [10]:
#Ver particiones actuales
print("Particiones antes de reparticionar:", df_features.rdd.getNumPartitions())

#Reparticionar
df_particionado = df_features.repartition(8)
print("Particiones después de reparticionar:", df_particionado.rdd.getNumPartitions())

#Guardar en Parquet
ruta_salida = Path("../Datos/Analiticos/Parquet/dataset_guia03_features.parquet")
(
    df_particionado.write
    .mode("overwrite")
    .parquet(str(ruta_salida.resolve()))
)

#Releer para confirmar que se guardó bien
df_verificacion = spark.read.parquet(str(ruta_salida.resolve()))
print("Filas leídas desde el nuevo Parquet:", df_verificacion.count())
print("Columnas:", len(df_verificacion.columns))

Particiones antes de reparticionar: 16
Particiones después de reparticionar: 8
Filas leídas desde el nuevo Parquet: 2827562
Columnas: 87
